# 맨해튼의 월세는 무엇으로 설명되고, 얼마나 정확하게 예측할 수 있을까?
### StreetEasy 임대 매물 데이터 기반 예측 모델 구축 · 포트폴리오 작성 가이드

**문서 상태: 분석 설계 및 작성 템플릿 / 실제 데이터 분석·모델 학습 미수행**

이 노트북은 보스턴 주택가격 분석 노트북의 구성과 서술 방식을 참고하여, 제공된 PDF·Markdown 과제지시서의 13개 미션을 포트폴리오 이야기로 재구성한 자료다. 과제를 이미 해결한 결과물이 아니라, **무엇을 분석하고 어떤 근거를 남기며 어떻게 설명할지**를 정리한 작업용 노트북이다.

- `[실행 후 작성]`과 대괄호 문구를 실제 결과로 교체한다. 지시서가 예고한 결과도 직접 확인하기 전에는 사실로 쓰지 않는다.
- 실행 가능한 시작 코드와 분석 보조 함수를 포함한다. 모델 학습·군집·SHAP 전 과정을 자동 실행하는 완성 코드는 포함하지 않는다.
- 각 절의 **과제 요구**와 **작성 제안**을 구분했다. 선택한 기준·모델 목록은 지시서가 지정한 정답이 아니다.
- 제출할 때는 안내 문구와 비어 있는 작성 칸을 정리하고, 코드 출력과 해석을 남긴다.

작성자: [이름] · 분석일: [날짜] · 데이터 수집 시점: [출처 확인 후 작성]


## 이 노트북을 사용하는 방법

1. #1~#2에서 분석 질문과 변수 정의를 확정한다.
2. #3~#4에서 관찰한 수치로 #5의 전처리 결정을 뒷받침한다.
3. #6에서 동일한 데이터 분할로 11종의 기준 성능과 튜닝 성능을 비교한다.
4. #7~#8에서 과적합·변수 축소·SHAP를 연결하여 최종 결론을 쓴다.

보스턴 파일의 장점은 결과표 뒤에 **‘그래서 무엇을 결정했는가’**가 있다는 점이다. 이번에도 표와 그래프를 나열하는 데서 멈추지 말고, 각 절 끝에 관찰 → 판단 → 결정 → 다음 단계의 네 문장을 쓴다.

| 참고 노트북의 구성 | 이번 포트폴리오 구성 | 대응 미션 |
|---|---|---|
| #1 분석 개요 | #1 분석 목적·질문·평가 설계 | 전체 |
| #2 데이터 소개 | #2 출처·분석 단위·변수·가설 | 미션 1 준비 |
| #3 품질 점검 및 기본 전처리 | #3 품질 점검·기술통계 | 1~2 |
| #4 탐색적 데이터 분석 | #4 변수 관계·정보 중복 | 3~4 |
| #5 모델링 | #5 파생·권역·변수 확정, #6 모델 비교·튜닝 | 5~9 |
| #6 모형 진단과 개선 | #7 과적합·변수 축소 | 10~11 |
| #7 결과 해석 | #8 SHAP·개별 예측·다섯 줄 결론 | 12~13 |

**참고 파일에서 그대로 옮기지 않을 것:** 보스턴 데이터의 변수·수치·최종 모델, 전체 데이터에 적합한 학습 오차, 회귀계수 중심 해석이다. 이번 과제는 여러 예측 모델을 비교하므로 검증 성능과 SHAP가 중심이다. 관측자료의 연관성을 인과관계처럼 서술하지 않는다.


## #1. 분석 개요
### 1-1. 목적과 질문

**목적 문장 예시**

> 맨해튼 임대 매물의 주거 특성·편의시설·입지 정보를 이용해 월 임대료를 예측한다. 여러 모델의 예측 오차와 과적합 정도를 비교하고, 최종 모델이 개별 매물에 부여한 가격의 근거와 적용 한계를 설명한다.

| 분석 질문 | 필요한 근거 | 답변 위치 |
|---|---|---|
| 면적·공간 구성·편의시설 중 월세와 관련이 큰 특성은 무엇인가? | 상관·집단 검정·효과크기 | #4 |
| 동네를 소수 권역으로 요약해도 위치 정보를 보존할 수 있는가? | 월세를 제외한 군집 프로파일·권역 구성 | #5 |
| 어떤 모델이 오차·안정성·비용을 함께 만족하는가? | 11종 비교·5겹 교차검증·학습 시간 | #6~#7 |
| 변수를 줄여도 성능을 유지하는가? | 축소 전후 동일 조건 비교 | #7 |
| 특정 매물의 월세가 높거나 낮게 예측되는 이유는 무엇인가? | SHAP·대표 매물 3건·USD 잔차 | #8 |

`주지표는 **RMSE(USD/월)**, 보조지표는 MAE(USD/월), R²로 제안한다. RMSE는 큰 오차에 더 민감하고, MAE는 평균적인 금액 오차를 설명하기 쉽다. 실제 거래가격이 아닌 매물 호가 데이터인지 출처에서 확인한다.`


### 1-2. 보고 순서와 실행 순서를 구분하기

지시서에는 미션 7에서 전처리를 마친 뒤 미션 8에서 분할하라고 되어 있지만, 감점 기준에는 **검증 데이터를 전처리 기준 산출에 쓰지 말 것**이 명시되어 있다. 다음 설계는 이 충돌을 피하기 위한 작성 제안이다.

1. 원본 구조·중복·결측·값의 범위를 기록한다. 중복 처리 등 데이터 정체성에 관한 규칙을 명시한다.
2. 본격적인 분포 기반 판단과 전처리 학습 전에 **80:20, 시드 42**로 행 ID를 고정한다.
3. 미션 2~7의 변수 선택과 변환 판단은 훈련 자료로 진행한다. 미션 8에서는 이미 저장한 분할을 불러온다.
4. 중앙값 대체·스케일링·희소 수준 기준·동네 평균·군집·학습형 변수 선택은 **교차검증의 각 훈련 폴드에서 다시 학습**한다. 전체 80%에서 전처리한 파일을 그대로 5겹 검증에 넣지 않는다.
5. 20% 검증 자료의 오차는 비교 보고용으로 남기되, 튜닝·변수 선택은 훈련 내부 교차검증으로 정한다. 검증 결과를 보고 모델을 계속 수정했다면 이 자료는 개발용 검증셋이므로, 최종 성능 확인에는 별도 테스트 자료가 필요하다고 쓴다.

각 미션의 서술 순서는 유지하되, 실제 분할을 앞당긴 이유를 보고서에 한 문장으로 남긴다. 전체 표본에 대한 빈도 보고와 모델 학습용 통계량 산출은 구분한다.


### 1-3. 환경 설정

아래 코드는 시작용이다. `pandas`, `numpy`, `matplotlib`, `scipy`, `scikit-learn`, `hossam`이 설치된 환경에서 실행한다. 추가 분석에는 `statsmodels`, `shap` 등이 필요하다. 이 가이드에서는 패키지를 설치하거나 외부 데이터를 내려받지 않았다.


In [ ]:
from pathlib import Path
import sys
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split, KFold

SEED = 42
ARTIFACTS = Path("manhattan_artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Python:", sys.version)
for package in ["pandas", "numpy", "scipy", "scikit-learn", "hossam"]:
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "설치 필요")

def save_table(table, name):
    table.to_csv(ARTIFACTS / f"{name}.csv", encoding="utf-8-sig", index=True)
    display(table)


## #2. 데이터 소개
### 2-1. 출처와 분석 단위

**과제 제공 정보:** StreetEasy 맨해튼 임대 매물, 약 3,500건, 특정 시점의 스냅숏이다. 정확한 행 수·시점·열 이름은 로딩 결과로 확인한다. 지시서의 32개 동네·579건/1건 같은 수치는 확인 대상이지 실행 결과가 아니다.

다음 항목을 기재한다: 제공 경로(`hossam`), 원출처, 수집 시점, 관측 단위, 대상 지역, `rent`의 단위, 호가/실거래 구분, 원자료 크기, 이용 조건. 확인되지 않은 항목은 ‘확인되지 않음’으로 남긴다.


In [ ]:
# 과제지시서에 제시된 데이터 로딩 방식
from hossam import load_data
origin = load_data("streeteasy")
if not isinstance(origin, pd.DataFrame):
    raise TypeError("로딩 결과가 DataFrame인지 먼저 확인하세요.")
df = origin.copy(deep=True)
print("원본 크기:", df.shape)
display(df.head())
display(pd.DataFrame({"dtype": df.dtypes.astype(str)}))


### 2-2. 변수 정의서와 초기 가설

아래 열 이름과 단위는 **로딩 설명에 대조할 작업 초안**이다. 실제 데이터와 다르면 먼저 수정한다. 식별자는 설명변수에서 제외한다.

| 변수 | 의미·확인할 단위 | 분석상 유형 | 초기 처리 |
|---|---|---|---|
| rent | 월 임대료, USD/월 확인 | 연속형 목표 | 분포 확인 후 log 또는 log1p 판단 |
| size_sqft | 전용 면적, ft² 확인 | 연속형 | 0·음수 확인 |
| bedrooms / bathrooms | 침실·욕실 수 | 수치형 | 0의 의미 별도 확인 |
| floor | 층 | 수치형 | 0이 지상층 등인지 확인 |
| building_age_yrs | 건물 연수 | 수치형 | 0이 신축인지 확인 |
| min_to_subway | 역까지 소요 시간, 분 확인 | 수치형 | 음수·0의 의미 확인 |
| no_fee | 수수료 없음 여부 | 이진 범주형 | 편의시설 합계에는 제외 |
| has_* 7종 | 개별 편의시설 보유 여부 | 이진 범주형 | 각각 정의 확인 |
| neighborhood | 동네 | 명목형 | 희소 병합·권역화 검토 |
| borough 등 | 자치구 등 지역 구분 | 명목형 | 상수인지 실제 확인 |

**초기 가설:** H1 면적이 클수록 월세가 높다. H2 역까지 시간이 길수록 월세가 낮다. H3 편의시설 보유 매물의 월세가 높다. H4 권역에 따라 월세 분포가 다르다. 모두 검증 전 가설이며, 단변량 관계와 다른 변수를 고려한 예측 기여가 다를 수 있다.


## #3. 품질 점검 및 기술통계
### 3-1. 표본 구조·품질 점검 — 미션 1 · 5점

**과제 요구:** 이진 8개 변수를 범주형으로 확정하고 행·열·자료형, 완전 중복, 결측, 최솟값·최댓값, 0의 의미, 명목형 빈도 및 상수 변수를 기록한다. 이 단계에서는 이상값을 발견했다고 바로 삭제하지 않는다.

**작성 제안:** 침실 0은 스튜디오일 수 있고 건물 연수 0은 신축일 수 있다. 욕실·층의 0도 오류라고 단정하지 말고 정의를 확인한다. 동일 특성의 매물이 실제로 서로 다른 집일 수 있으므로, 완전 중복의 삭제 여부에도 근거를 남긴다.


In [ ]:
binary_cols = [c for c in df.columns if c == "no_fee" or c.startswith("has_")]
print("이진 변수 후보:", binary_cols, "개수:", len(binary_cols))
for c in binary_cols:
    values = set(df[c].dropna().unique())
    if not values.issubset({0, 1}):
        raise ValueError(f"{c}: 0/1 이외의 값 확인 필요: {values}")
    df[c] = df[c].astype("category")
for c in ["neighborhood", "borough"]:
    if c in df:
        df[c] = df[c].astype("category")

quality = pd.DataFrame({"dtype": df.dtypes.astype(str),
                        "missing": df.isna().sum(),
                        "levels": df.nunique(dropna=False)})
save_table(quality, "m01_quality")
print("완전 중복 행 수:", df.duplicated().sum())
print("상수 변수:", df.columns[df.nunique(dropna=False).eq(1)].tolist())
numeric = df.select_dtypes(include="number")
save_table(numeric.agg(["min", "max"]).T, "m01_ranges")
for c in ["bedrooms", "bathrooms", "floor", "building_age_yrs"]:
    if c in df:
        print(c, "0인 행:", df[c].eq(0).sum())
for c in df.select_dtypes(include=["category", "object", "string"]):
    save_table(df[c].value_counts(dropna=False).to_frame("count"), f"m01_frequency_{c}")


**분석 후 작성할 문장**

0을 유지하거나 보류한 이유와, 상수 변수가 매물 간 차이를 설명할 수 없는 이유를 적는다. 발견 사항 표의 열은 `변수 / 현상 / 건수 / 의미 / 처리 예정 / 근거`로 만든다.

> 관찰: [표·그림의 실제 수치]  
> 판단: [그 수치가 뜻하는 것과 불확실성]  
> 결정: [유지·제외·변환·추가 검증]  
> 다음 단계: [이번 결정이 다음 분석에 미치는 영향]


### 3-2. 분할을 먼저 고정하기

아래 코드는 원자료 보존을 기본으로 한다. 중복 삭제가 타당하다고 확인했다면 옵션을 바꾸고 근거를 기록한다. 반복 매물·동일 건물 ID가 있으면 동일 집단이 양쪽에 섞이지 않도록 그룹 분할을 검토한다. 중복을 그대로 나누면 동일 기록이 훈련과 검증에 들어갈 수 있다.


In [ ]:
DROP_EXACT_DUPLICATES = False  # 데이터 정체성 확인 후 결정
base = df.drop_duplicates().copy() if DROP_EXACT_DUPLICATES else df.copy()
base = base.reset_index(drop=True)
base.index.name = "row_id"
train_ids, valid_ids = train_test_split(base.index.to_numpy(), test_size=0.2, random_state=SEED)
train_raw = base.loc[train_ids].copy()
valid_raw = base.loc[valid_ids].copy()
split = pd.Series("train", index=base.index, name="split")
split.loc[valid_ids] = "validation"
save_table(split.to_frame(), "split_manifest")
assert set(train_ids).isdisjoint(valid_ids)
print("훈련/검증:", len(train_raw), len(valid_raw))
# 이하 분포 기반 의사결정은 train_raw에서 시작한다.


### 3-3. 기술통계와 변환 후보 — 미션 2 · 5점

**과제 요구:** 평균·중앙값·왜도·첨도·IQR 이상치 비율을 함께 읽고, `rent` 분포 및 로그 후보의 우선순위와 이유를 남긴다. 여기서는 변환하지 않는다.

**해석 원칙:** IQR 밖의 값은 오류와 동의어가 아니다. 고가 매물이 실제 관측이면 임의 삭제가 예측 대상 범위를 바꾼다. 이진형 평균은 보유율이므로 연속형 왜도 해석에 섞지 않는다. 목표변수의 정규분포는 모든 예측 모델의 필수 조건이 아니다. 로그 변환은 꼬리·이분산·예측 손실을 고려해 판단한다.


In [ ]:
def describe_numeric(frame):
    x = frame.select_dtypes(include="number")
    q1, q3 = x.quantile(.25), x.quantile(.75)
    iqr = q3 - q1
    out = (x.lt(q1 - 1.5 * iqr) | x.gt(q3 + 1.5 * iqr)).sum()
    table = x.describe().T
    table["median"] = x.median()
    table["skew"] = x.skew()
    table["excess_kurtosis"] = x.kurt()  # 정규분포 기준 0
    table["iqr_outlier_pct"] = 100 * out / x.count().replace(0, np.nan)
    table["transform_decision"] = "실행 후 작성"
    return table

save_table(describe_numeric(train_raw), "m02_statistics")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
train_raw["rent"].dropna().hist(bins=40, ax=axes[0])
axes[0].set(xlabel="Rent (USD/month)", ylabel="Count")
axes[1].boxplot(train_raw["rent"].dropna(), vert=False)
axes[1].set(xlabel="Rent (USD/month)")
fig.tight_layout()
fig.savefig(ARTIFACTS / "m02_rent_distribution.png", dpi=160)
plt.show()


**분석 후 작성할 문장**

평균·중앙값 차이, 왜도, 첨도, 이상치 비율이 같은 변수를 가리키는지 적고, `변수 / 변환 후보 순위 / 근거 / 0·음수 존재 여부` 표를 만든다.

> 관찰: [표·그림의 실제 수치]  
> 판단: [그 수치가 뜻하는 것과 불확실성]  
> 결정: [유지·제외·변환·추가 검증]  
> 다음 단계: [이번 결정이 다음 분석에 미치는 영향]


## #4. 탐색적 데이터 분석
### 4-1. 월세와 변수의 관계 — 미션 3 · 7점

| 대상 | 수행할 분석 | 남길 표·그림 | 해석할 내용 |
|---|---|---|---|
| 수치형 독립변수 | 산점도 및 Pearson 또는 Spearman | 변수·방법·계수·p값·표본 수, 절대 계수 순 정렬 | 분포뿐 아니라 선형/단조 관계를 보고 방법 선택 |
| 이진 변수 8개 | 집단별 분포·정규성·분산 점검 후 검정 | 집단별 n·평균·중앙값, 검정명·p값·효과크기 | 통계적 유의성과 금액 차이의 실질적 크기 |
| neighborhood | ANOVA 및 사후검정 | 집단 요약·전체 검정·전체 쌍별 비교표 | 많은 비교를 본문에서 어떻게 요약할 것인가 |

평균 차이가 질문이면 Welch t 검정을 검토한다. Mann–Whitney U를 선택하면 이를 ‘평균 차이 검정’이라고 쓰지 않는다. t 검정에는 Hedges g 등 표준화 평균 차이, U 검정에는 순위 기반 효과크기처럼 방법에 맞는 지표를 붙인다. 큰 표본에서는 정규성 검정만으로 방법을 자동 결정하지 않는다.

ANOVA 가정과 집단 크기를 확인하고, 등분산을 가정하는 Tukey 또는 이분산을 고려한 대안을 선택한다. 희소 집단의 추론 한계를 남긴다. 32수준이면 496쌍이지만, 훈련 자료에서 관측된 수준이 다르면 `k*(k-1)/2`에 맞춰 보고한다. 전체 쌍별 표는 파일로, 본문은 대표 비교와 요약으로 남긴다.

지시서가 예고한 ‘차이가 없는 편의시설’은 실제 검정으로 찾는다. **비유의 = 효과가 없다는 증명**은 아니며, 비유의 하나만으로 예측 변수를 자동 삭제하지 않는다. 여러 검정의 p값 보정 방식도 기록한다.


In [ ]:
from scipy import stats

def correlation_table(frame, columns, target="rent", method="spearman"):
    if method not in {"pearson", "spearman"}:
        raise ValueError("pearson 또는 spearman을 지정하세요.")
    rows = []
    for c in columns:
        pair = frame[[c, target]].replace([np.inf, -np.inf], np.nan).dropna()
        if len(pair) < 3 or pair[c].nunique() < 2 or pair[target].nunique() < 2:
            rows.append([c, method, len(pair), np.nan, np.nan])
            continue
        fn = stats.spearmanr if method == "spearman" else stats.pearsonr
        r, p = fn(pair[c], pair[target])
        rows.append([c, method, len(pair), r, p])
    result = pd.DataFrame(rows, columns=["variable", "method", "n", "coefficient", "p_value"])
    return result.assign(abs_coefficient=result.coefficient.abs()).sort_values("abs_coefficient", ascending=False)

# 미션 2와 산점도를 확인한 뒤 method를 선택하여 호출한다.
# cols = [c for c in train_raw.select_dtypes("number") if c != "rent"]
# save_table(correlation_table(train_raw, cols, method="spearman"), "m03_correlations")


**분석 후 작성할 문장**

가장 관련이 큰 변수, 효과가 작거나 불확실한 편의시설, 동네 쌍별 비교를 그대로 제시하기 어려운 이유를 각각 적는다.

> 관찰: [표·그림의 실제 수치]  
> 판단: [그 수치가 뜻하는 것과 불확실성]  
> 결정: [유지·제외·변환·추가 검증]  
> 다음 단계: [이번 결정이 다음 분석에 미치는 영향]


### 4-2. 독립변수 간 중복 — 미션 4 · 6점

**과제 요구:** 수치형 독립변수의 상관 히트맵, 서로 겹치는 세 변수, 대표 변수 선정, 버리는 정보의 보존 방법, 1순위 투입 변수의 근거를 남긴다.

**검토 후보:** 면적·침실 수·욕실 수는 공간 규모라는 공통 정보를 가질 수 있다. 이는 출발 가설이며 실제 행렬로 확인한다. 강한 상관은 선형계수의 불안정성을 키울 수 있지만 모든 모델의 성능을 반드시 악화시키는 것은 아니다.

| 변수 묶음 | 관측 상관계수 | 대표 후보 | 남길 이유 | 보존할 정보·파생 아이디어 |
|---|---|---|---|---|
| [실행 후 작성] | [실행 후 작성] | [실행 후 작성] | 목표 관련성·측정 안정성 | 방 하나당 면적 등 |

**1순위 변수 보고 예시:** ‘[변수]는 월세와 [계수]의 관계를 보였다. [다른 변수]와 정보가 겹치지만 [측정·해석 측면 장점]이 있다. 따라서 대표로 남기고 공간 구성 정보는 [파생변수]로 보완한다.’


## #5. 파생변수 설계 및 투입 변수 확정
### 5-1. 연속형 파생변수 — 미션 5 · 9점

**과제 요구:** 1차 파생 4종 이상 → 분모·결측·무한대 확인 → 정리 → 2차 파생 1종을 순서대로 수행한다.

| 제안 이름 | 계산식 | 의미 | 학습할 기준·주의 |
|---|---|---|---|
| area_per_room | size_sqft / (bedrooms + bathrooms) | 공간 구성 대비 면적 | 합이 0인 행 확인 |
| amenity_count | has_* 7종 합계 | 편의시설 다양성 | no_fee 제외, 결측을 0으로 간주하지 않기 |
| relative_area | size_sqft / 동네 평균 size_sqft | 동네 내 상대적 크기 | 각 훈련 폴드에서 평균 계산 |
| relative_floor | floor / 동네 평균 floor | 동네 내 상대적 층 | 동네 평균이 0일 때 처리 |
| relative_area_per_room (2차) | 정리한 area_per_room / 동네 평균 area_per_room | 동네 대비 공간 여유 | 1차 정리 후 평균 계산 |

편의시설 합계는 엄밀히 이산 계수형이다. 과제의 수치형 파생 예시로 사용하되, 연속형과 구별해서 설명한다. 추가 아이디어로 `size_sqft / (bedrooms + 1)`을 검토할 수 있지만 +1의 의미와 중복 검증을 남겨야 한다.

지시서의 무한대 처리 기준은 **5% 미만 해당 행 제거 / 5% 이상 결측으로 바꾼 뒤 대체 판단**이다. 이는 이 과제의 규칙이며 일반적인 통계 법칙은 아니다. 행 기준 비율의 분모·분자와 변수별 건수를 함께 남긴다. `0/0`은 inf가 아닌 NaN이므로 별도로 센다.

행 제거 규칙은 훈련 자료에서 정하고 이후 고정한다. 검증에서 예측할 수 없는 행을 제외한다면 제외 건수·예측 가능 비율을 공개하고 모든 모델을 같은 행 집합에서 비교한다. 전처리의 행 제거는 일반 Pipeline에서 y와 어긋날 수 있으므로 분할별 X/y를 함께 처리하는 별도 절차가 필요하다.

**누수 방지 구현 설계:** 동네 평균 변환기는 `fit(X_train)`에서 평균과 전체 평균 대체값을 저장하고, `transform(X_valid)`에서는 저장값만 사용한다. 미관측 동네는 훈련 전체 평균 등 사전에 정한 값으로 처리한다. 교차검증 안에서도 같은 동작을 반복한다.


**분석 후 작성할 문장**

`파생변수 / 분모 최솟값 / inf 건수·비율 / NaN 건수 / 처리 / 전후 행 수` 표를 만들고, 1차 값을 정리한 뒤 2차를 만든 이유를 설명한다.

> 관찰: [표·그림의 실제 수치]  
> 판단: [그 수치가 뜻하는 것과 불확실성]  
> 결정: [유지·제외·변환·추가 검증]  
> 다음 단계: [이번 결정이 다음 분석에 미치는 영향]


### 5-2. 동네를 권역으로 재설계 — 미션 6 · 10점

**순서**

1. 각 훈련 폴드에서 동네 빈도를 세고 **1% 미만** 수준을 `Other`로 병합한다. 정확히 1%는 병합 대상이 아니다.
2. 병합된 동네별로 면적·층·건물 연수·역까지 시간·편의시설 보유율 등 **X만으로 프로파일**을 만든다.
3. 프로파일의 결측 처리와 표준화를 훈련 자료에서 수행한 뒤 군집을 학습한다.
4. 후보 군집 수별 실루엣 점수, 군집 크기, 해석 가능성을 비교한다. 후보 k는 프로파일 수보다 작아야 한다.
5. 동네 → 권역 매핑을 고정한다. 검증의 희소·미관측 동네는 훈련에서 정한 `Other` 또는 별도 미지정 정책으로 처리한다.
6. 권역 확정 후에만 권역별 매물 수·평균 월세를 요약한다. 평균 월세는 **사후 해석용**이며, 군집 수나 이름의 우열을 골라 모델을 바꾸는 기준으로 사용하지 않는다.

| 산출물 | 본문에서 답할 것 |
|---|---|
| 동네별 매물 수 그림·병합 전후 수준 수 | 희소 수준을 왜 묶었나? |
| k별 지표 표·군집 크기 | 가장 높은 점수의 k를 그대로 선택했나? |
| 권역·구성 동네·매물 수·평균 월세 표 | 실제 특성이 일관된 묶음인가? |
| 권역 프로파일 표와 명칭 | ‘대형·시설 풍부’처럼 X 특성으로 설명 가능한가? |

**범주형 파생 1종 이상:** 예를 들어 `near_subway = min_to_subway <= 5`를 제안한다. 5분은 가정한 분석 기준이며 공인된 기준이라고 쓰지 않는다. 도메인 근거를 밝히거나 5/10분 기준을 훈련 내부에서 비교한다.

동네가 32개라고 해서 원핫 인코딩이 불가능한 것은 아니다. 이 과제에서 권역화하는 이유는 희소성과 설명 복잡도를 줄이기 위해서라고 쓴다.


**분석 후 작성할 문장**

병합 전후 수준 수, 선택한 k와 이유, 권역별 이름과 구성, 새 매물의 미관측 동네 처리 정책을 적는다.

> 관찰: [표·그림의 실제 수치]  
> 판단: [그 수치가 뜻하는 것과 불확실성]  
> 결정: [유지·제외·변환·추가 검증]  
> 다음 단계: [이번 결정이 다음 분석에 미치는 영향]


### 5-3. 파생 검증·변수 확정·저장 — 미션 7 · 10점

모든 파생변수에 #4와 같은 검증을 반복한다. 수치형은 상관, 2집단은 집단 비교, 다집단은 ANOVA·사후검정, 파생끼리는 중복 점검을 수행한다. 예상과 반대 부호가 나오면 분모 효과·집단 구성·다른 변수와의 중복을 검토한다.

| 변수 | 원본/파생 | 관련성·효과크기 | 중복·누수 점검 | 채택/제외 | 한 줄 사유 |
|---|---|---|---|---|---|
| [실행 후 작성] | [유형] | [수치] | [근거] | [결정] | [사유] |

**변환 및 저장 규칙**

- 로그 대상과 이유를 명시한다. `log`는 양수, `log1p`는 이 과제에서 비음수 변수에 적용한다. 0을 허용하는지 먼저 확인한다. 목표변수도 검토·변환하고, 원본 rent를 별도로 보존한다.
- CSV의 정수 라벨은 저장 방식일 뿐이다. 다시 읽을 때 범주형으로 복원하고, 모델 입력에서는 원핫 인코딩 등으로 처리한다. 권역 2가 권역 1보다 두 배 크다는 뜻이 아니다.
- `before_transform.csv`, `after_log_label.csv` 두 상태를 남긴다. 각각 row_id·split을 포함하되 이 둘은 설명변수에 넣지 않는다. 범주 매핑·로그 대상·원본 단위는 `preprocessing_schema.json`에 저장한다.
- 이미 변환된 전체 파일은 검토용 체크포인트다. 교차검증은 원본 X와 폴드 안의 전처리기를 사용한다. 저장 파일을 통해 재시작하려면 원시 열과 전처리 설정도 복원 가능하게 남긴다.
- 전역 p값으로 변수를 고른 뒤 수행한 CV는 완전히 독립적인 선택 후 성능 추정이 아니다. 데이터로 학습하는 선택은 폴드 안으로 넣고, 엄밀한 선택 성능은 중첩 CV 또는 별도 테스트로 확인한다.


## #6. 모델 비교 및 튜닝
### 6-1. 11종 베이스라인 — 미션 8 · 9점

**과제 요구:** 같은 80:20 분할과 같은 지표로 11종을 비교한다. 지시서에 정확한 모델명은 없으므로 아래는 구성 제안이다. 수업에서 지정한 목록이 있다면 그 목록으로 교체한다.

| 모델 | 필요한 전처리 | 튜닝할 설정 예시 |
|---|---|---|
| LinearRegression | 수치 대체·스케일, 범주 원핫 | fit_intercept |
| Ridge | 동일 | alpha |
| Lasso | 동일 | alpha, max_iter |
| ElasticNet | 동일 | alpha, l1_ratio |
| KNeighborsRegressor | 스케일 필수적으로 검토·원핫 | n_neighbors, weights, p |
| SVR | 스케일·원핫 | C, epsilon, gamma |
| DecisionTreeRegressor | 대체·원핫 | max_depth, min_samples_leaf |
| RandomForestRegressor | 대체·원핫 | n_estimators, max_features, min_samples_leaf |
| ExtraTreesRegressor | 대체·원핫 | n_estimators, max_features, min_samples_leaf |
| GradientBoostingRegressor | 대체·원핫 | n_estimators, learning_rate, max_depth |
| AdaBoostRegressor | 대체·원핫 | n_estimators, learning_rate, loss |

별도로 `DummyRegressor`를 추가하면 단순 평균·중앙값 예측과 비교할 수 있다. Dummy는 위 11종에 포함하지 않는다. 로그 목표를 쓸 때 USD 평균을 내는 Dummy와 로그 평균을 역변환한 Dummy는 다르므로 정의를 적는다.

**성능표 열:** 모델 / 순위 / CV RMSE 평균·표준편차 / 검증 RMSE / 검증 MAE / 검증 R² / 학습 시간 / 선형 대비 차이.

해석할 질문: 선형보다 낮은 모델은 무엇인가? 단일 트리와 숲의 차이는 얼마인가? 1위와 근소한 차이의 상위 그룹은 무엇인가? ‘근소’는 예를 들어 RMSE 1% 이내와 CV 변동폭을 함께 보는 운영 기준으로 사전 정의한다. 통계적 동등성을 증명한 표현은 쓰지 않는다.


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def regression_metrics_usd(y_true_usd, y_pred_usd):
    """입력은 모두 역변환이 끝난 USD/월이어야 한다."""
    y_true_usd = np.asarray(y_true_usd, dtype=float)
    y_pred_usd = np.asarray(y_pred_usd, dtype=float)
    if not (np.isfinite(y_true_usd).all() and np.isfinite(y_pred_usd).all()):
        raise ValueError("평가값의 결측·무한대를 먼저 확인하세요.")
    return {"RMSE_USD": np.sqrt(mean_squared_error(y_true_usd, y_pred_usd)),
            "MAE_USD": mean_absolute_error(y_true_usd, y_pred_usd),
            "R2": r2_score(y_true_usd, y_pred_usd)}

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
# 구현 예: TransformedTargetRegressor(regressor=전처리와_모델_Pipeline,
#                                    func=np.log1p, inverse_func=np.expm1)
# 위 구조는 y_train을 USD로 전달하고 predict()도 USD를 반환한다.
# 이미 로그 변환한 y를 위 구조에 다시 전달하면 이중 변환이므로 금지한다.


### 6-2. 5겹 교차검증과 튜닝 — 미션 9 · 8점

미션 8에서는 집중 검토할 계열을 선정하되, **미션 9에서는 11종 모두** 탐색한다. 선형회귀처럼 탐색 범위가 작은 모델도 탐색한 설정과 한계를 기록한다.

**작성할 절차:** 동일한 5개 폴드 → 전처리와 모델을 하나의 학습 단위로 구성 → 모델별 탐색 범위 고정 → CV RMSE로 최적 조합 선정 → 고정 검증셋 평가 → 소요 시간과 개선량 비교.

군집을 미리 전체 훈련 자료에서 만들어 놓으면 CV 폴드 경계를 넘는다. 대체·스케일링뿐 아니라 동네 평균, 빈도 병합, 군집도 전처리기의 `fit`에 포함한다. 검증셋을 조기 종료나 반복 실험의 기준으로 사용하지 않는다.

| 모델 | 기본 CV RMSE | 튜닝 CV RMSE | 최적 설정 | 검증 전/후 RMSE | 개선율 | 시간 |
|---|---|---|---|---|---|---|
| [11행 작성] | [값] | [값] | [설정] | [값] | [값] | [초] |

개선율(%) = `(기본 RMSE - 튜닝 RMSE) / 기본 RMSE × 100`으로 계산한다. 기본과 튜닝 모두 같은 단위·동일 평가셋으로 계산해야 한다. 탐색에 사용한 CV의 최적 점수는 선택 편향이 있을 수 있으므로 독립적인 테스트 성능으로 표현하지 않는다.

**결정 문장 틀:** ‘[모델]의 RMSE는 [전]에서 [후]로 [비율]% 감소했지만 추가 학습에 [시간]이 들었다. [사전 기준·변동폭·운영 비용]을 고려하여 추가 튜닝을 [진행/중단]한다.’


## #7. 과적합 진단과 모델 단순화
### 7-1. 외운 모델을 가려내기 — 미션 10 · 9점

상위 모델별로 훈련·교차검증·고정 검증 성능을 나란히 둔다. 훈련/CV 비교는 가능하면 같은 CV 실행에서 나온 폴드별 train score와 held-out score를 쓴다.

**판정 기준 제안:** `RMSE 격차율 = (CV RMSE - 훈련 RMSE) / CV RMSE × 100`. 예를 들어 20% 초과를 경고 기준으로 미리 정할 수 있다. 20%는 지시서의 지정값이나 보편적인 법칙이 아니며, 실제 값·모델 특성·다른 지표를 함께 판단한다. R²는 음수·0 근처에서 비율이 불안정하므로 훈련-CV의 차이로 보고한다.

| 모델 | 훈련 RMSE | CV RMSE±SD | 검증 RMSE | RMSE 격차율 | MAE 격차 | R² 차이 | 시간 | 채택/탈락 이유 |
|---|---|---|---|---|---|---|---|---|
| [상위 모델] | [값] | [값] | [값] | [값] | [값] | [값] | [값] | [근거] |

훈련 1위가 검증에서도 1위인지, 일부 지표만 경고하는지 설명한다. 실제로 격차가 큰 모델이 없으면 탈락 사례를 만들어내지 않는다. 최종 선택은 오차 → 안정성 → 시간·해석 가능성 등 우선순위를 명시한다.


### 7-2. 중요도 95% 기준으로 축소 — 미션 11 · 8점

1. 선택 모델에 맞는 중요도를 구하고 계산 방법을 적는다. 트리 중요도, 표준화 계수, permutation importance는 서로 의미가 다르다.
2. 중요도를 내림차순으로 정규화하고 누적 비율을 계산한다. **95%를 처음 넘어서는 변수까지 포함**한다. 원핫 열을 원본 변수로 합산할지, 변환된 열 단위로 셀지 명시한다.
3. 탈락 변수와 이유를 쓰고, 축소 후 원시 입력으로 다시 학습·다시 튜닝한다. 학습형 중요도 선택도 CV 훈련 폴드 안에서 수행한다.
4. 축소 전후 오차·과적합·시간·필요 원시 입력 수를 비교한다. 성능 변화가 수용 범위를 벗어나면 원래 구성을 유지할 수 있다.

Permutation importance는 음수가 나올 수 있다. 누적 비율 계산을 위해 음수를 0으로 처리했다면 그 사실과 원래 값도 공개한다. 전부 0이면 95% 기준 자체가 의미 없으므로 다른 중요도·선택 방법을 검토한다.

**비용 해석의 핵심:** 최종 입력에 `area_per_room`만 남아도 이를 만들기 위해 면적·침실·욕실의 원시 정보가 필요하다. 파생변수 개수 감소를 곧바로 수집 비용 감소로 쓰지 않는다.

**최종 모델 문장:** ‘[모델명]을 [원본 입력 수]개 원시 항목에서 만든 [모델 입력 수]개 변수로 구성했다. 축소 전 대비 RMSE는 [차이]이고 [시간·설명 복잡도] 측면의 이익을 고려해 채택했다.’


## #8. 모델 해석과 최종 결론
### 8-1. SHAP 영향력·방향·변동성 — 미션 12 · 7점

**과제 요구:** 최종 모델의 SHAP를 계산하고 평균 절대기여, 평균 기여, 변동성 및 상위 두 변수 비중을 남긴다. 해석 대상 표본과 배경 자료의 출처·크기·시드를 기록한다. 배경 자료는 훈련 자료에서 구성하고, 설명 자료가 훈련인지 검증인지 밝힌다.

| 지표 | 계산 | 읽는 법 |
|---|---|---|
| 영향력 | mean(abs(SHAP)) | 크면 예측에 크게 기여 |
| 평균 방향 | mean(SHAP) | 해당 표본에서 기준 예측보다 평균적으로 올림/내림 |
| 변동성 | std(SHAP) | 매물별 기여가 얼마나 달라지는가 |
| 상위 2개 비중 | 상위 2개 영향력 합 / 전체 영향력 합 | 영향력이 집중된 정도 |

평균 SHAP가 0에 가깝더라도 절대기여가 크면 상승·하락이 상쇄된 것이다. 양의 평균 기여는 ‘변수값이 커질수록 월세가 오른다’와 다르다. 방향은 beeswarm과 dependence에서 확인한다.

**척도 주의:** 로그 월세를 출력하는 내부 모델을 설명하면 SHAP도 로그 척도다. SHAP 값을 각각 지수화하여 달러 기여로 더하면 안 된다. `log1p` 모델이라면 최종 합계 `base_value + SHAP 합`을 `expm1`하여 예측 월세를 복원한다. USD를 반환하는 전체 예측 함수를 직접 설명하는 방식이면 USD 기여로 해석하되 계산 비용을 기록한다.

원핫 권역을 합산해 원본 권역 기여로 표시할 때는 같은 매물의 해당 SHAP 열을 먼저 합산한다. 권역 숫자 라벨의 크기를 순서로 해석하지 않는다. 중요도와 SHAP는 인과효과를 증명하지 않는다.


### 8-2. 네 가지 그림과 대표 매물 — 미션 13 · 7점

| 그림 | 작성할 내용 | 해석 문장 틀 |
|---|---|---|
| Bar | 평균 절대 SHAP, 별도로 미션 11의 누적 중요도·95% 경계 | ‘[변수]의 영향력이 크며 축소 기준에서는 [변수]까지 유지했다.’ |
| Beeswarm | 변수값 색과 SHAP 부호 | ‘높은 값이 [방향]에 모이지만 [예외]가 관찰된다.’ |
| Dependence | 비단조 변수의 값과 SHAP, 필요 시 권역별 비교 | ‘[구간]에서 방향이 바뀌며 단일 방향 요약이 어렵다.’ |
| Waterfall | 실제 rent 기준 최저·중앙 순위·최고 매물 3건 | ‘기준 예측 [값]에서 [주요 기여]가 더해져 [예측값]이 되었다.’ |

미션 11의 선택 중요도와 SHAP 중요도는 다를 수 있다. 최종 모델의 SHAP Bar에는 이미 제외한 변수가 없으므로, 제외 경계를 보여주려면 축소 전 중요도 그림도 같이 둔다.

단색 점은 범주형 표현·값의 변동 부족·표시 방식 때문일 수 있다. 색만으로 방향을 읽을 수 없는 이유를 해당 데이터와 그림 설정에서 확인한다. 권역별 SHAP 평균·분포·표본 수로 가장 큰 하향 기여 권역을 찾고 권역명을 붙인다.

대표 매물은 ‘설명 대상 검증 표본에서 실제 rent를 정렬해 0, n//2, n-1번째’처럼 선택 규칙을 적는다. 중간은 중앙 순위 매물이며 평균 월세와 가장 가까운 매물과는 다르다. 3건을 설명하기 전에 그 사례를 보고 모델을 바꾸지 않는다.

| 매물 row_id | 구분 | 실제 USD | 예측 USD | 잔차(실제-예측) | 과대/과소 예측 | 주요 기여 |
|---|---|---|---|---|---|---|
| [ID] | 최저 | [값] | [값] | [값] | [판정] | [설명] |
| [ID] | 중앙 | [값] | [값] | [값] | [판정] | [설명] |
| [ID] | 최고 | [값] | [값] | [값] | [판정] | [설명] |

추가로 USD 잔차 대 예측값 그림과 가격 구간별 MAE를 제시하면 고가 매물의 과소예측 여부를 설명하기 좋다. 로그 예측을 단순 역변환한 값은 일반적으로 조건부 평균과 일치하지 않는다. 평균 월세 추정이 목적이라면 편향 보정의 필요성을 훈련 내부에서 검토한다.


### 8-3. 질문별 답변과 가설 판정

보스턴 참고 노트북처럼 시작한 질문으로 돌아와 답한다. 숫자는 실행 결과를 확인한 뒤 작성한다.

| 가설 | 지지/불확실/기각 | 단변량 근거 | 최종 모델에서의 근거 | 해석 한계 |
|---|---|---|---|---|
| H1. 면적이 클수록 월세가 높다 | [판정] | [계수·p·그림] | [SHAP 관계] | 인과로 해석하지 않음 |
| H2. 역까지 시간이 길수록 월세가 낮다 | [판정] | [근거] | [근거] | 다른 입지 특성과 중복 가능 |
| H3. 편의시설 보유 매물이 비싸다 | [판정] | [효과크기·금액 차이] | [근거] | 시설별 결과 구분 |
| H4. 권역에 따라 월세 분포가 다르다 | [판정] | [집단 검정] | [권역별 기여] | 표본 수·지역 범위 제한 |

**활용:** 표본과 비슷한 맨해튼 매물의 월세를 비교하는 학습용 예측 사례로 설명한다. **한계:** 특정 시점·지역의 표본, 관측되지 않은 건물 품질·계약 조건, 공간·동일 건물 의존성, 현재 시장으로의 일반화, 극단 가격대의 불확실성을 기록한다. 데이터가 허용하지 않는 실거래 가격·다른 도시·미래 시장 정확도를 주장하지 않는다.


### 8-4. 다섯 줄 결론 — 실제 분석 후 완성할 최종 요약

1. **최종 모델:** [모델명]을 [변수 수]개 모델 입력으로 구성했으며, [성능·안정성·비용의 선택 근거] 때문에 채택했다.
2. **성능:** 고정 검증셋의 RMSE는 [값] USD/월, MAE는 [값] USD/월이고, 기본 모델 대비 RMSE가 [값]% 개선되었다.
3. **월세의 구조:** [변수·조건]이 예측을 높이고 [변수·조건]이 낮추는 양상을 보였다. [비단조 변수]는 하나의 방향으로 요약하기 어려웠다.
4. **위험 관리:** [동네 평균·군집·변수 선택]의 학습을 훈련 폴드 내부로 제한했고, [과적합 진단과 대응]을 수행했다.
5. **믿을 수 있는 범위:** 이 결과는 [관측 시점]의 맨해튼 매물과 유사한 범위에 한정되며, 현재 시장·다른 지역·새 건물군에 적용하려면 별도 검증이 필요하다.

첫 화면에는 분석을 마친 뒤 이 다섯 줄을 복사해 요약으로 배치한다. 실행 전에는 성능이나 1위 변수를 추정해서 채우지 않는다.


## 부록 A. 제출 산출물 체크리스트

| 미션 | 배점 | 완료 확인 | 저장할 산출물 예시 |
|---|---:|---|---|
| 1 | 5 | ☐ 타입·중복·결측·범위·0·상수·빈도 | m01_quality, ranges, frequency, findings |
| 2 | 5 | ☐ 네 지표·rent 분포·로그 근거 | m02_statistics, rent_distribution |
| 3 | 7 | ☐ 상관·이진 효과크기·ANOVA·전체 사후검정 | m03_correlations, binary_tests, posthoc |
| 4 | 6 | ☐ 상관 히트맵·세 변수 중복·대표 이유 | m04_heatmap, representative_decision |
| 5 | 9 | ☐ 1차 4종·inf/NaN·5% 처리·2차 파생 | m05_feature_definitions, invalid_audit |
| 6 | 10 | ☐ 1% 병합·X만 군집·k 근거·권역명·범주 파생 | m06_cluster_scores, region_profiles |
| 7 | 10 | ☐ 파생 검증·채택표·로그·라벨·두 파일 | before_transform, after_log_label, schema |
| 8 | 9 | ☐ 같은 분할·11종 표·계열 해석 | m08_baseline, split_manifest |
| 9 | 8 | ☐ 5겹 CV·11종 튜닝·최적값·지속 판단 | m09_tuning, best_parameters |
| 10 | 9 | ☐ 훈련/CV/검증 격차·다기준 선정 | m10_overfit_diagnosis |
| 11 | 8 | ☐ 95% 경계·축소 재학습·재튜닝·비용 | m11_importance, reduction_comparison |
| 12 | 7 | ☐ 크기·방향·변동성·평균 0 해석 | m12_shap_summary |
| 13 | 7 | ☐ 네 그림·권역·3매물·다섯 줄·한계 | m13_shap_plots, case_predictions |

가산점 항목: 직접 설계하고 검증한 파생변수(+2), USD 역변환 잔차 해석(+2), 누수 위험 발견·회피 기록(+1). 가산점보다 필수 산출물과 판단 근거를 먼저 완성한다.

**큰 감점 방지 확인:** 목표변수로 입력을 만들지 않았는가(-15)? 검증 자료로 전처리·튜닝하지 않았는가(-10)? 범주 정수를 연속적인 크기로 학습시키지 않았는가(-5)? 미션 간 분할이 바뀌지 않았는가(-5)?

마지막으로 커널을 재시작하여 위에서부터 실행하고, 오류·미작성 칸·실행 순서 의존성을 확인한다. 표·그래프와 본문 수치, 최종 변수 수, 단위가 모두 일치해야 한다.


## 부록 B. 자료 확인 범위와 이번 파일의 역할

- **구성 참고:** `boston-housing-linear-regression.ipynb`의 분석 개요·데이터 소개·품질 점검·EDA·모델링·진단·결과 해석 구조와 근거 중심 서술.
- **요구사항 근거:** 제공된 `[LAB-03] 머신러닝-예측모형ㅣ캡스톤 - 뉴욕 맨해튼 임대료 예측 모델 구축ㅣ과제지시서.pdf`(10쪽) 및 같은 이름의 Markdown. 13개 미션, 배점, 산출물, 가산·감점 기준을 대조하여 반영했다.
- 두 지시서의 미션 7에는 ‘14번 미션’ 언급이 있으나 실제 본문과 채점표는 13개 미션으로 구성된다. 이 가이드는 실제 목록인 13개를 따른다.
- 분할 시점 조정, 모델 11종의 구체적 목록, RMSE 주지표, 20% 과적합 경고 예시, 5분 역세권 예시는 **이 가이드의 작성 제안**이다.
- 이번 작업은 사용자가 요청한 작성 방향 정리다. 첨부 지시서의 과제 수행 요구를 현재 작업에서 모두 실행하라는 명령으로 해석하지 않았다. 실제 StreetEasy 데이터 로딩·전처리·검정·학습·SHAP 결과는 만들지 않았으며, 포함 코드는 시작용 예시와 보조 함수다.
